In [1]:
# =============================================================================
# RoBERTa Fine-Tuning for Fake/Real News Detection
# Optimized for RTX A6000 (fp32 stable mode, patience=3, under 12-15 hours)
# Similar to your provided notebook
# =============================================================================

# ────────────────────────────────────────────────────────────────
# Cell 1: Quick environment check (updated for A6000 48GB)
# ────────────────────────────────────────────────────────────────

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device() if torch.cuda.is_available() else "cpu")
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CUDA in PyTorch:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else "N/A")
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else "N/A")

# Clear cache for fresh start
torch.cuda.empty_cache()

PyTorch: 2.6.0+cu124
CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA RTX A6000
CUDA in PyTorch: 12.4
cuDNN: 90100
VRAM (GB): 51.526500352


In [2]:
# ────────────────────────────────────────────────────────────────
# Cell 2: Imports
# ────────────────────────────────────────────────────────────────

import numpy as np
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

c:\Users\Dr-ShalluSharma\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ────────────────────────────────────────────────────────────────
# Cell 3: Load datasets (modify paths here)
# ────────────────────────────────────────────────────────────────

# Modify these paths to your actual files
train_path = "train.csv"  # ← MODIFY: your train CSV path
val_path = "val.csv"      # ← MODIFY: your val CSV path
test_path = "test.csv"    # ← MODIFY: your test CSV path

# Load as HF datasets
tokenized_datasets = load_dataset(
    "csv",
    data_files={
        "train": train_path,
        "validation": val_path,
        "test": test_path
    }
)

In [4]:
# ────────────────────────────────────────────────────────────────
# Cell 4: Tokenization and data preparation (modify max_length if needed)
# ────────────────────────────────────────────────────────────────

model_name = "roberta-base"  # ← MODIFY: use "roberta-large" if you want better but slower model
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)  # ← MODIFY: max_length=256 if texts are short

tokenized_datasets = tokenized_datasets.map(tokenize_function, batched=True)

Map: 100%|██████████| 71984/71984 [00:16<00:00, 4291.70 examples/s]


In [5]:
# ────────────────────────────────────────────────────────────────
# Cell 5: Load model (modify num_labels if more classes)
# ────────────────────────────────────────────────────────────────

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # ← MODIFY: 2 for binary (real/fake); change if more classes
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
# ────────────────────────────────────────────────────────────────
# Cell 6: Metrics function (already includes accuracy, precision, recall, F1)
# ────────────────────────────────────────────────────────────────

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }

In [7]:
# ────────────────────────────────────────────────────────────────
# Cell 7: Training arguments (modify batch sizes, epochs, etc.)
# ────────────────────────────────────────────────────────────────

training_args = TrainingArguments(
    output_dir="results",  # ← MODIFY: folder for checkpoints
    evaluation_strategy="epoch",
    learning_rate=2e-5,    # ← MODIFY: lower to 1e-5 if unstable
    per_device_train_batch_size=64,  # ← MODIFY: 32–128 based on VRAM
    per_device_eval_batch_size=128,  # ← MODIFY: higher for faster eval
    num_train_epochs=7,              # ← MODIFY: your 7 epochs
    weight_decay=0.01,
    fp16=True,                       # ← MODIFY: False if unstable
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_strategy="epoch",
    logging_dir="logs",
    logging_steps=500,
    save_total_limit=3,
)

c:\Users\Dr-ShalluSharma\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [8]:
# ────────────────────────────────────────────────────────────────
# Cell 8: Trainer setup (modify patience)
# ────────────────────────────────────────────────────────────────

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,       # ← same as your notebook
            early_stopping_threshold=0.0005
        )
    ],
)

C:\Users\Dr-ShalluSharma\AppData\Local\Temp\ipykernel_278384\2635126715.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# dont close model Training 

In [9]:
# ────────────────────────────────────────────────────────────────
# Cell 9: Train (same call, but optimized params will finish under 12 hours)
# ────────────────────────────────────────────────────────────────

print("Starting fine-tuning...")
trainer.train()

Starting fine-tuning...


  1%|▏         | 500/36743 [04:23<4:46:10,  2.11it/s]

{'loss': 0.1529, 'grad_norm': 4.679154396057129, 'learning_rate': 1.972947228043437e-05, 'epoch': 0.1}


  3%|▎         | 1000/36743 [08:45<5:25:18,  1.83it/s]

{'loss': 0.0919, 'grad_norm': 2.805910587310791, 'learning_rate': 1.9457311596766734e-05, 'epoch': 0.19}


...

                                                       
100%|██████████| 36743/36743 [5:12:28<00:00,  2.25it/s]

{'eval_loss': 0.11445251852273941, 'eval_accuracy': 0.9851216937097133, 'eval_f1': 0.9851216127409027, 'eval_precision': 0.985131634356474, 'eval_recall': 0.9851216937097133, 'eval_runtime': 160.7597, 'eval_samples_per_second': 447.774, 'eval_steps_per_second': 3.502, 'epoch': 7.0}


100%|██████████| 36743/36743 [5:12:30<00:00,  1.96it/s]

{'train_runtime': 18750.3594, 'train_samples_per_second': 125.409, 'train_steps_per_second': 1.96, 'train_loss': 0.025608580984002642, 'epoch': 7.0}


TrainOutput(global_step=36743, training_loss=0.025608580984002642, metrics={'train_runtime': 18750.3594, 'train_samples_per_second': 125.409, 'train_steps_per_second': 1.96, 'total_flos': 6.186972271252685e+17, 'train_loss': 0.025608580984002642, 'epoch': 7.0})

# dont close model training

In [10]:
# ────────────────────────────────────────────────────────────────
# Cell 10: Test evaluation (same)
# ────────────────────────────────────────────────────────────────

print("\nFinal test evaluation...")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\nTest results:")
for k, v in test_results.items():
    if k.startswith("eval_"):
        print(f"{k.replace('eval_', '').ljust(12)} : {v:.4f}")


Final test evaluation...


100%|██████████| 563/563 [02:42<00:00,  3.46it/s]


Test results:
loss         : 0.1007
accuracy     : 0.9846
f1           : 0.9846
precision    : 0.9846
recall       : 0.9846
runtime      : 163.2401
samples_per_second : 440.9700
steps_per_second : 3.4490


In [11]:
# ────────────────────────────────────────────────────────────────
# Cell 11: Save model (updated path)
# ────────────────────────────────────────────────────────────────

save_path = "./roberta-fake-news-best-aug"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved → {save_path}")

Model saved → ./roberta-fake-news-best-aug


In [12]:
import pandas as pd

log_history = trainer.state.log_history
df = pd.DataFrame(log_history)

df.to_csv("training_log.csv", index=False)
print("Saved to training_log.csv")
print(df.to_string())

Saved to training_log.csv
      loss  grad_norm  learning_rate     epoch   step  eval_loss  eval_accuracy   eval_f1  eval_precision  eval_recall  eval_runtime  eval_samples_per_second  eval_steps_per_second  train_runtime  train_samples_per_second  train_steps_per_second    total_flos  train_loss
0   0.1529   4.679154   1.972947e-05  0.095256    500        NaN            NaN       NaN             NaN          NaN           NaN                      NaN                    NaN            NaN                       NaN                     NaN           NaN         NaN
1   0.0919   2.805911   1.945731e-05  0.190512   1000        NaN            NaN       NaN             NaN          NaN           NaN                      NaN                    NaN            NaN                       NaN                     NaN           NaN         NaN
2   0.0798   1.290696   1.918515e-05  0.285769   1500        NaN            NaN       NaN             NaN          NaN           NaN                      NaN 